In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import average_precision_score


# ============================================================
# 0. 데이터 준비
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

X_array = X_cn7.to_numpy(dtype=np.float32)
y_array = y_cn7.to_numpy()

input_dim = X_array.shape[1]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# OOF 결과 저장
oof_logistic_prob = np.zeros(len(X_array))
oof_mahalanobis_score = np.zeros(len(X_array))
oof_vae_score = np.zeros(len(X_array))


# ============================================================
# 1. VAE 정의
# ============================================================

class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar
        )

        reconstructed = self.decoder(z)

        return reconstructed, mu, logvar


def vae_loss(
    reconstructed,
    x,
    mu,
    logvar,
    beta=0.01
):

    reconstruction_loss = torch.mean(
        (reconstructed - x) ** 2
    )

    kl_loss = -0.5 * torch.mean(
        1 + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    return reconstruction_loss + beta * kl_loss


# ============================================================
# 2. 5-Fold OOF
# Logistic + Mahalanobis + VAE
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    print(f"Fold {fold}")

    X_train = X_array[train_idx]
    X_val = X_array[val_idx]

    y_train = y_array[train_idx]

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------

    logistic = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    logistic.fit(
        X_train,
        y_train
    )

    oof_logistic_prob[val_idx] = (
        logistic.predict_proba(X_val)[:, 1]
    )


    # --------------------------------------------------------
    # Mahalanobis
    # 정상 데이터만 학습
    # --------------------------------------------------------

    X_train_normal = X_train[
        y_train == 0
    ]

    lw = LedoitWolf()

    lw.fit(
        X_train_normal
    )

    diff = (
        X_val
        - lw.location_
    )

    mahal_sq = np.einsum(
        "ij,jk,ik->i",
        diff,
        lw.precision_,
        diff
    )

    oof_mahalanobis_score[val_idx] = np.sqrt(
        np.maximum(
            mahal_sq,
            0
        )
    )


    # --------------------------------------------------------
    # VAE
    # 정상 데이터만 학습
    # --------------------------------------------------------

    torch.manual_seed(
        42 + fold
    )

    train_tensor = torch.tensor(
        X_train_normal,
        dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=64,
        shuffle=True
    )

    vae = VAE(
        input_dim=input_dim,
        latent_dim=8
    )

    optimizer = torch.optim.Adam(
        vae.parameters(),
        lr=0.001
    )

    vae.train()

    for epoch in range(100):

        for (batch,) in train_loader:

            optimizer.zero_grad()

            reconstructed, mu, logvar = vae(
                batch
            )

            loss = vae_loss(
                reconstructed,
                batch,
                mu,
                logvar,
                beta=0.01
            )

            loss.backward()

            optimizer.step()


    # --------------------------------------------------------
    # VAE Validation Score
    # --------------------------------------------------------

    vae.eval()

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32
    )

    with torch.no_grad():

        mu, logvar = vae.encode(
            X_val_tensor
        )

        reconstructed = vae.decoder(
            mu
        )

        reconstruction_error = torch.mean(
            (
                X_val_tensor
                - reconstructed
            ) ** 2,
            dim=1
        )

    oof_vae_score[val_idx] = (
        reconstruction_error
        .cpu()
        .numpy()
    )


# ============================================================
# 3. 단일 모델 PR-AUC
# ============================================================

print("\n[OOF PR-AUC]")

print(
    "Logistic:",
    average_precision_score(
        y_array,
        oof_logistic_prob
    )
)

print(
    "Mahalanobis:",
    average_precision_score(
        y_array,
        oof_mahalanobis_score
    )
)

print(
    "VAE:",
    average_precision_score(
        y_array,
        oof_vae_score
    )
)


# ============================================================
# 4. 앙상블 DataFrame 생성
# ============================================================

ensemble_df = pd.DataFrame({
    "Target": y_array,
    "Logistic": oof_logistic_prob,
    "Mahalanobis": oof_mahalanobis_score,
    "VAE": oof_vae_score
})


# ============================================================
# 5. 각 점수를 0~1로 변환
# ============================================================

scaler = MinMaxScaler()

ensemble_df[
    [
        "Logistic_norm",
        "Mahalanobis_norm",
        "VAE_norm"
    ]
] = scaler.fit_transform(
    ensemble_df[
        [
            "Logistic",
            "Mahalanobis",
            "VAE"
        ]
    ]
)


# ============================================================
# 6. 단순 평균 앙상블
# ============================================================

ensemble_df["Ensemble_Score"] = (
    ensemble_df["Logistic_norm"]
    + ensemble_df["Mahalanobis_norm"]
    + ensemble_df["VAE_norm"]
) / 3


# ============================================================
# 7. PR-AUC 비교
# ============================================================

scores = {
    "Logistic": average_precision_score(
        y_array,
        oof_logistic_prob
    ),

    "Mahalanobis": average_precision_score(
        y_array,
        oof_mahalanobis_score
    ),

    "VAE": average_precision_score(
        y_array,
        oof_vae_score
    ),

    "Ensemble": average_precision_score(
        y_array,
        ensemble_df["Ensemble_Score"]
    )
}


result_df = pd.DataFrame(
    scores.items(),
    columns=[
        "Model",
        "PR-AUC"
    ]
).sort_values(
    "PR-AUC",
    ascending=False
)

print("\n[Model Comparison]")

display(result_df)


# ============================================================
# 8. Ensemble Rank
# ============================================================

ensemble_df["Ensemble_Rank"] = (
    ensemble_df["Ensemble_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)


# 불량만 확인
defect_rank_df = (
    ensemble_df[
        ensemble_df["Target"] == 1
    ][
        [
            "Target",
            "Logistic_norm",
            "Mahalanobis_norm",
            "VAE_norm",
            "Ensemble_Score",
            "Ensemble_Rank"
        ]
    ]
    .sort_values(
        "Ensemble_Rank"
    )
)


print("\n[Defect Ensemble Rank]")

display(defect_rank_df)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

[OOF PR-AUC]
Logistic: 0.46941459686939674
Mahalanobis: 0.4207075406767129
VAE: 0.4231035325996964

[Model Comparison]


,Model,PR-AUC
0,Logistic,0.469415
3,Ensemble,0.439217
2,VAE,0.423104
1,Mahalanobis,0.420708



[Defect Ensemble Rank]


,Target,Logistic_norm,Mahalanobis_norm,VAE_norm,Ensemble_Score,Ensemble_Rank
117,1,0.915870,1.000000,0.887130,0.934333,1.0
118,1,0.973208,0.828652,1.000000,0.933953,2.0
119,1,0.973584,0.822303,0.973395,0.923094,3.0
120,1,0.974642,0.813309,0.971975,0.919976,4.0
116,1,1.000000,0.933512,0.809162,0.914225,5.0
115,1,0.978842,0.773789,0.938917,0.897183,6.0
95,1,0.081709,0.123037,0.167867,0.124204,26.0
109,1,0.148738,0.079297,0.022032,0.083356,37.0
47,1,0.143503,0.050548,0.027496,0.073849,44.0
113,1,0.031589,0.084083,0.069068,0.061580,56.0


In [5]:
from sklearn.metrics import average_precision_score
import pandas as pd

results = []

# Logistic / Mahalanobis / VAE 가중치 탐색
for w_log in [
    0.5, 0.6, 0.7, 0.8, 0.9
]:

    remaining = 1 - w_log

    for w_maha_ratio in [
        0.0, 0.25, 0.5, 0.75, 1.0
    ]:

        w_maha = remaining * w_maha_ratio
        w_vae = remaining - w_maha

        score = (
            w_log * ensemble_df["Logistic_norm"]
            + w_maha * ensemble_df["Mahalanobis_norm"]
            + w_vae * ensemble_df["VAE_norm"]
        )

        pr_auc = average_precision_score(
            y_array,
            score
        )

        results.append({
            "Logistic_Weight": w_log,
            "Mahalanobis_Weight": w_maha,
            "VAE_Weight": w_vae,
            "PR-AUC": pr_auc
        })


weight_result_df = pd.DataFrame(
    results
).sort_values(
    "PR-AUC",
    ascending=False
).reset_index(drop=True)


display(
    weight_result_df.head(15)
)

,Logistic_Weight,Mahalanobis_Weight,VAE_Weight,PR-AUC
0,0.9,0.025,0.075,0.467122
1,0.9,0.000,0.100,0.466999
2,0.9,0.075,0.025,0.466880
3,0.9,0.050,0.050,0.466747
4,0.9,0.100,0.000,0.466454
5,0.8,0.050,0.150,0.465587
6,0.8,0.000,0.200,0.464992
7,0.8,0.100,0.100,0.464543
8,0.8,0.150,0.050,0.464503
9,0.8,0.200,0.000,0.462799


In [6]:
pair_results = []

for w in [
    0.5, 0.6, 0.7, 0.8, 0.9
]:

    # Logistic + Mahalanobis
    score_lm = (
        w * ensemble_df["Logistic_norm"]
        + (1 - w) * ensemble_df["Mahalanobis_norm"]
    )

    pair_results.append({
        "Combination": "Logistic + Mahalanobis",
        "Logistic_Weight": w,
        "Other_Weight": 1 - w,
        "PR-AUC": average_precision_score(
            y_array,
            score_lm
        )
    })


    # Logistic + VAE
    score_lv = (
        w * ensemble_df["Logistic_norm"]
        + (1 - w) * ensemble_df["VAE_norm"]
    )

    pair_results.append({
        "Combination": "Logistic + VAE",
        "Logistic_Weight": w,
        "Other_Weight": 1 - w,
        "PR-AUC": average_precision_score(
            y_array,
            score_lv
        )
    })


pair_result_df = pd.DataFrame(
    pair_results
).sort_values(
    "PR-AUC",
    ascending=False
).reset_index(drop=True)


display(pair_result_df)

,Combination,Logistic_Weight,Other_Weight,PR-AUC
0,Logistic + VAE,0.9,0.1,0.466999
1,Logistic + Mahalanobis,0.9,0.1,0.466454
2,Logistic + VAE,0.8,0.2,0.464992
3,Logistic + Mahalanobis,0.8,0.2,0.462799
4,Logistic + VAE,0.7,0.3,0.460230
5,Logistic + VAE,0.6,0.4,0.457573
6,Logistic + Mahalanobis,0.7,0.3,0.456283
7,Logistic + VAE,0.5,0.5,0.452823
8,Logistic + Mahalanobis,0.6,0.4,0.451545
9,Logistic + Mahalanobis,0.5,0.5,0.448715


Mahalanobis Nested CV

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


# ============================================================
# 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# Mahalanobis score 함수
# ============================================================

def mahalanobis_score(model, X):

    diff = X - model.location_

    mahal_sq = np.einsum(
        "ij,jk,ik->i",
        diff,
        model.precision_,
        diff
    )

    return np.sqrt(
        np.maximum(mahal_sq, 0)
    )


# ============================================================
# Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


outer_scores = np.zeros(len(X_array))
outer_preds = np.zeros(
    len(X_array),
    dtype=int
)

fold_results = []


# ============================================================
# Nested CV
# ============================================================

for outer_fold, (
    outer_train_idx,
    outer_val_idx
) in enumerate(
    outer_cv.split(
        X_array,
        y_array
    ),
    start=1
):

    print(
        f"\n========== Outer Fold {outer_fold} =========="
    )

    X_outer_train = X_array[
        outer_train_idx
    ]

    y_outer_train = y_array[
        outer_train_idx
    ]

    X_outer_val = X_array[
        outer_val_idx
    ]

    y_outer_val = y_array[
        outer_val_idx
    ]


    # ========================================================
    # Inner CV
    # ========================================================

    inner_cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + outer_fold
    )

    inner_oof_score = np.zeros(
        len(X_outer_train)
    )


    for inner_train_idx, inner_val_idx in (
        inner_cv.split(
            X_outer_train,
            y_outer_train
        )
    ):

        X_inner_train = X_outer_train[
            inner_train_idx
        ]

        y_inner_train = y_outer_train[
            inner_train_idx
        ]

        X_inner_val = X_outer_train[
            inner_val_idx
        ]


        # 정상만 사용
        X_inner_normal = X_inner_train[
            y_inner_train == 0
        ]


        # Mahalanobis 학습
        model = LedoitWolf()

        model.fit(
            X_inner_normal
        )


        # Inner validation score
        inner_oof_score[
            inner_val_idx
        ] = mahalanobis_score(
            model,
            X_inner_val
        )


    # ========================================================
    # Inner OOF에서 threshold 결정
    # ========================================================

    candidate_thresholds = np.unique(
        inner_oof_score
    )

    best_threshold = None
    best_f1 = -1


    for threshold in candidate_thresholds:

        inner_pred = (
            inner_oof_score >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_outer_train,
            inner_pred,
            zero_division=0
        )

        if current_f1 > best_f1:

            best_f1 = current_f1
            best_threshold = threshold


    print(
        "Selected Threshold:",
        best_threshold
    )

    print(
        "Inner F1:",
        best_f1
    )


    # ========================================================
    # Outer Train 정상 데이터로 최종 Mahalanobis 학습
    # ========================================================

    X_outer_normal = X_outer_train[
        y_outer_train == 0
    ]

    final_model = LedoitWolf()

    final_model.fit(
        X_outer_normal
    )


    # ========================================================
    # Outer Validation 평가
    # ========================================================

    val_score = mahalanobis_score(
        final_model,
        X_outer_val
    )

    val_pred = (
        val_score >= best_threshold
    ).astype(int)


    outer_scores[
        outer_val_idx
    ] = val_score

    outer_preds[
        outer_val_idx
    ] = val_pred


    tn, fp, fn, tp = confusion_matrix(
        y_outer_val,
        val_pred,
        labels=[0, 1]
    ).ravel()


    fold_results.append({
        "Fold": outer_fold,
        "Threshold": best_threshold,
        "Inner_F1": best_f1,

        "Outer_Precision": precision_score(
            y_outer_val,
            val_pred,
            zero_division=0
        ),

        "Outer_Recall": recall_score(
            y_outer_val,
            val_pred,
            zero_division=0
        ),

        "Outer_F1": f1_score(
            y_outer_val,
            val_pred,
            zero_division=0
        ),

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


# ============================================================
# Fold별 결과
# ============================================================

nested_maha_df = pd.DataFrame(
    fold_results
)

print(
    "\n[Nested CV Fold Results]"
)

display(
    nested_maha_df
)


# ============================================================
# 전체 Outer OOF 성능
# ============================================================

precision = precision_score(
    y_array,
    outer_preds,
    zero_division=0
)

recall = recall_score(
    y_array,
    outer_preds,
    zero_division=0
)

f1 = f1_score(
    y_array,
    outer_preds,
    zero_division=0
)

pr_auc = average_precision_score(
    y_array,
    outer_scores
)

tn, fp, fn, tp = confusion_matrix(
    y_array,
    outer_preds,
    labels=[0, 1]
).ravel()


print(
    "\n[Nested CV Overall]"
)

print(
    "PR-AUC   :", pr_auc
)

print(
    "Precision:", precision
)

print(
    "Recall   :", recall
)

print(
    "F1       :", f1
)

print(
    "TP       :", tp
)

print(
    "FP       :", fp
)

print(
    "FN       :", fn
)

print(
    "TN       :", tn
)


========== Outer Fold 1 ==========
Selected Threshold: 27.75322522645123
Inner F1: 0.47058823529411764

========== Outer Fold 2 ==========
Selected Threshold: 26.266772863490317
Inner F1: 0.5263157894736842

========== Outer Fold 3 ==========
Selected Threshold: 26.198761362007627
Inner F1: 0.5263157894736842

========== Outer Fold 4 ==========
Selected Threshold: 30.133040887111566
Inner F1: 0.5714285714285714

========== Outer Fold 5 ==========
Selected Threshold: 6.204925359915403
Inner F1: 0.4444444444444444

[Nested CV Fold Results]


,Fold,Threshold,Inner_F1,Outer_Precision,Outer_Recall,Outer_F1,TP,FP,FN,TN
0,1,27.753225,0.470588,1.000000,0.500000,0.666667,2,0,2,239
1,2,26.266773,0.526316,1.000000,0.333333,0.500000,1,0,2,239
2,3,26.198761,0.526316,1.000000,0.333333,0.500000,1,0,2,239
3,4,30.133041,0.571429,0.000000,0.000000,0.000000,0,0,3,239
4,5,6.204925,0.444444,0.285714,0.500000,0.363636,2,5,2,233



[Nested CV Overall]
PR-AUC   : 0.4207075406767129
Precision: 0.5454545454545454
Recall   : 0.35294117647058826
F1       : 0.42857142857142855
TP       : 6
FP       : 5
FN       : 11
TN       : 1189


VAE Nested CV

In [8]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


# ============================================================
# 0. 데이터
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

X_array = X_cn7.to_numpy(dtype=np.float32)
y_array = y_cn7.to_numpy()

input_dim = X_array.shape[1]


# ============================================================
# 1. VAE
# ============================================================

class VAE(nn.Module):

    def __init__(self, input_dim, latent_dim=8):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )


    def encode(self, x):

        h = self.encoder(x)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar


    def reparameterize(
        self,
        mu,
        logvar
    ):

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(std)

        return mu + eps * std


    def forward(self, x):

        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar
        )

        reconstructed = self.decoder(z)

        return reconstructed, mu, logvar


# ============================================================
# 2. Loss
# ============================================================

def vae_loss(
    reconstructed,
    x,
    mu,
    logvar,
    beta=0.01
):

    reconstruction_loss = torch.mean(
        (reconstructed - x) ** 2
    )

    kl_loss = -0.5 * torch.mean(
        1
        + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    return (
        reconstruction_loss
        + beta * kl_loss
    )


# ============================================================
# 3. VAE 학습 함수
# ============================================================

def train_vae(
    X_normal,
    seed,
    epochs=100
):

    torch.manual_seed(seed)

    train_tensor = torch.tensor(
        X_normal,
        dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=64,
        shuffle=True
    )

    model = VAE(
        input_dim=input_dim,
        latent_dim=8
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        for (batch,) in loader:

            optimizer.zero_grad()

            reconstructed, mu, logvar = model(
                batch
            )

            loss = vae_loss(
                reconstructed,
                batch,
                mu,
                logvar,
                beta=0.01
            )

            loss.backward()

            optimizer.step()

    return model


# ============================================================
# 4. Reconstruction Error
# ============================================================

def get_vae_score(
    model,
    X
):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    with torch.no_grad():

        # 평가에서는 random sampling 사용 X
        mu, logvar = model.encode(
            X_tensor
        )

        reconstructed = model.decoder(
            mu
        )

        score = torch.mean(
            (
                X_tensor
                - reconstructed
            ) ** 2,
            dim=1
        )

    return (
        score
        .cpu()
        .numpy()
    )


# ============================================================
# 5. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

outer_scores = np.zeros(
    len(X_array)
)

outer_preds = np.zeros(
    len(X_array),
    dtype=int
)

fold_results = []


# ============================================================
# 6. Nested CV
# ============================================================

for outer_fold, (
    outer_train_idx,
    outer_val_idx
) in enumerate(
    outer_cv.split(
        X_array,
        y_array
    ),
    start=1
):

    print(
        f"\n========== Outer Fold {outer_fold} =========="
    )

    X_outer_train = X_array[
        outer_train_idx
    ]

    y_outer_train = y_array[
        outer_train_idx
    ]

    X_outer_val = X_array[
        outer_val_idx
    ]

    y_outer_val = y_array[
        outer_val_idx
    ]


    # ========================================================
    # Inner CV
    # ========================================================

    inner_cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + outer_fold
    )

    inner_oof_score = np.zeros(
        len(X_outer_train)
    )


    for inner_fold, (
        inner_train_idx,
        inner_val_idx
    ) in enumerate(
        inner_cv.split(
            X_outer_train,
            y_outer_train
        ),
        start=1
    ):

        X_inner_train = X_outer_train[
            inner_train_idx
        ]

        y_inner_train = y_outer_train[
            inner_train_idx
        ]

        X_inner_val = X_outer_train[
            inner_val_idx
        ]


        # 정상만 학습
        X_inner_normal = X_inner_train[
            y_inner_train == 0
        ]


        # VAE 학습
        model = train_vae(
            X_inner_normal,
            seed=(
                1000
                + outer_fold * 10
                + inner_fold
            )
        )


        # Inner Validation score
        inner_oof_score[
            inner_val_idx
        ] = get_vae_score(
            model,
            X_inner_val
        )


    # ========================================================
    # 7. Inner OOF에서 F1 최대 Threshold 선택
    # ========================================================

    candidate_thresholds = np.unique(
        inner_oof_score
    )

    best_threshold = None
    best_f1 = -1


    for threshold in candidate_thresholds:

        inner_pred = (
            inner_oof_score
            >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_outer_train,
            inner_pred,
            zero_division=0
        )

        if current_f1 > best_f1:

            best_f1 = current_f1
            best_threshold = threshold


    print(
        "Selected Threshold:",
        best_threshold
    )

    print(
        "Inner F1:",
        best_f1
    )


    # ========================================================
    # 8. Outer Train 정상 데이터로 VAE 재학습
    # ========================================================

    X_outer_normal = X_outer_train[
        y_outer_train == 0
    ]

    final_model = train_vae(
        X_outer_normal,
        seed=2000 + outer_fold
    )


    # ========================================================
    # 9. Outer Validation
    # ========================================================

    val_score = get_vae_score(
        final_model,
        X_outer_val
    )

    val_pred = (
        val_score
        >= best_threshold
    ).astype(int)


    outer_scores[
        outer_val_idx
    ] = val_score

    outer_preds[
        outer_val_idx
    ] = val_pred


    tn, fp, fn, tp = confusion_matrix(
        y_outer_val,
        val_pred,
        labels=[0, 1]
    ).ravel()


    fold_results.append({

        "Fold": outer_fold,

        "Threshold":
            best_threshold,

        "Inner_F1":
            best_f1,

        "Outer_Precision":
            precision_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_Recall":
            recall_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_F1":
            f1_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


# ============================================================
# 10. Fold별 결과
# ============================================================

nested_vae_df = pd.DataFrame(
    fold_results
)

print(
    "\n[Nested VAE Fold Results]"
)

display(
    nested_vae_df
)


# ============================================================
# 11. 전체 Nested OOF 결과
# ============================================================

precision = precision_score(
    y_array,
    outer_preds,
    zero_division=0
)

recall = recall_score(
    y_array,
    outer_preds,
    zero_division=0
)

f1 = f1_score(
    y_array,
    outer_preds,
    zero_division=0
)

pr_auc = average_precision_score(
    y_array,
    outer_scores
)

tn, fp, fn, tp = confusion_matrix(
    y_array,
    outer_preds,
    labels=[0, 1]
).ravel()


print(
    "\n[Nested VAE Overall]"
)

print(
    "PR-AUC   :", pr_auc
)

print(
    "Precision:", precision
)

print(
    "Recall   :", recall
)

print(
    "F1       :", f1
)

print(
    "TP       :", tp
)

print(
    "FP       :", fp
)

print(
    "FN       :", fn
)

print(
    "TN       :", tn
)


========== Outer Fold 1 ==========
Selected Threshold: 5.933681964874268
Inner F1: 0.42105263157894735

========== Outer Fold 2 ==========
Selected Threshold: 5.508762359619141
Inner F1: 0.5

========== Outer Fold 3 ==========
Selected Threshold: 5.3834004402160645
Inner F1: 0.47619047619047616

========== Outer Fold 4 ==========
Selected Threshold: 5.208093166351318
Inner F1: 0.6

========== Outer Fold 5 ==========
Selected Threshold: 5.602627277374268
Inner F1: 0.4444444444444444

[Nested VAE Fold Results]


,Fold,Threshold,Inner_F1,Outer_Precision,Outer_Recall,Outer_F1,TP,FP,FN,TN
0,1,5.933682,0.421053,1.0,0.250000,0.400000,1,0,3,239
1,2,5.508762,0.500000,1.0,0.333333,0.500000,1,0,2,239
2,3,5.383400,0.476190,1.0,0.333333,0.500000,1,0,2,239
3,4,5.208093,0.600000,0.0,0.000000,0.000000,0,0,3,239
4,5,5.602627,0.444444,1.0,0.500000,0.666667,2,0,2,238



[Nested VAE Overall]
PR-AUC   : 0.4279957108860388
Precision: 1.0
Recall   : 0.29411764705882354
F1       : 0.45454545454545453
TP       : 5
FP       : 0
FN       : 12
TN       : 1194


좋습니다. 이제 Logistic Regression도 동일한 Nested CV 구조로 threshold

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

outer_probs = np.zeros(
    len(X_array)
)

outer_preds = np.zeros(
    len(X_array),
    dtype=int
)

fold_results = []


# ============================================================
# 2. Nested CV
# ============================================================

for outer_fold, (
    outer_train_idx,
    outer_val_idx
) in enumerate(
    outer_cv.split(
        X_array,
        y_array
    ),
    start=1
):

    print(
        f"\n========== Outer Fold {outer_fold} =========="
    )

    X_outer_train = X_array[
        outer_train_idx
    ]

    y_outer_train = y_array[
        outer_train_idx
    ]

    X_outer_val = X_array[
        outer_val_idx
    ]

    y_outer_val = y_array[
        outer_val_idx
    ]


    # ========================================================
    # 3. Inner CV
    # ========================================================

    inner_cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + outer_fold
    )

    inner_oof_prob = np.zeros(
        len(X_outer_train)
    )


    for (
        inner_train_idx,
        inner_val_idx
    ) in inner_cv.split(
        X_outer_train,
        y_outer_train
    ):

        X_inner_train = X_outer_train[
            inner_train_idx
        ]

        y_inner_train = y_outer_train[
            inner_train_idx
        ]

        X_inner_val = X_outer_train[
            inner_val_idx
        ]


        model = LogisticRegression(
            max_iter=1000,
            random_state=42
        )

        model.fit(
            X_inner_train,
            y_inner_train
        )


        inner_oof_prob[
            inner_val_idx
        ] = model.predict_proba(
            X_inner_val
        )[:, 1]


    # ========================================================
    # 4. Inner OOF에서 F1 최대 Threshold 선택
    # ========================================================

    candidate_thresholds = np.unique(
        inner_oof_prob
    )

    best_threshold = None
    best_f1 = -1


    for threshold in candidate_thresholds:

        inner_pred = (
            inner_oof_prob >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_outer_train,
            inner_pred,
            zero_division=0
        )

        if current_f1 > best_f1:

            best_f1 = current_f1
            best_threshold = threshold


    print(
        "Selected Threshold:",
        best_threshold
    )

    print(
        "Inner F1:",
        best_f1
    )


    # ========================================================
    # 5. 전체 Outer Train으로 Logistic 재학습
    # ========================================================

    final_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    final_model.fit(
        X_outer_train,
        y_outer_train
    )


    # ========================================================
    # 6. Outer Validation 확률
    # ========================================================

    val_prob = final_model.predict_proba(
        X_outer_val
    )[:, 1]

    val_pred = (
        val_prob >= best_threshold
    ).astype(int)


    outer_probs[
        outer_val_idx
    ] = val_prob

    outer_preds[
        outer_val_idx
    ] = val_pred


    # ========================================================
    # 7. Fold 성능
    # ========================================================

    tn, fp, fn, tp = confusion_matrix(
        y_outer_val,
        val_pred,
        labels=[0, 1]
    ).ravel()


    fold_results.append({

        "Fold": outer_fold,

        "Threshold":
            best_threshold,

        "Inner_F1":
            best_f1,

        "Outer_Precision":
            precision_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_Recall":
            recall_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_F1":
            f1_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


# ============================================================
# 8. Fold별 결과
# ============================================================

nested_logistic_df = pd.DataFrame(
    fold_results
)

print(
    "\n[Nested Logistic Fold Results]"
)

display(
    nested_logistic_df
)


# ============================================================
# 9. 전체 Outer OOF 성능
# ============================================================

precision = precision_score(
    y_array,
    outer_preds,
    zero_division=0
)

recall = recall_score(
    y_array,
    outer_preds,
    zero_division=0
)

f1 = f1_score(
    y_array,
    outer_preds,
    zero_division=0
)

pr_auc = average_precision_score(
    y_array,
    outer_probs
)

tn, fp, fn, tp = confusion_matrix(
    y_array,
    outer_preds,
    labels=[0, 1]
).ravel()


print(
    "\n[Nested Logistic Overall]"
)

print(
    "PR-AUC   :", pr_auc
)

print(
    "Precision:", precision
)

print(
    "Recall   :", recall
)

print(
    "F1       :", f1
)

print(
    "TP       :", tp
)

print(
    "FP       :", fp
)

print(
    "FN       :", fn
)

print(
    "TN       :", tn
)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

[OOF PR-AUC]
Logistic: 0.46941459686939674
Mahalanobis: 0.4207075406767129
VAE: 0.4231035325996964

[Model Comparison]


,Model,PR-AUC
0,Logistic,0.469415
3,Ensemble,0.439217
2,VAE,0.423104
1,Mahalanobis,0.420708



[Defect Ensemble Rank]


,Target,Logistic_norm,Mahalanobis_norm,VAE_norm,Ensemble_Score,Ensemble_Rank
117,1,0.915870,1.000000,0.887130,0.934333,1.0
118,1,0.973208,0.828652,1.000000,0.933953,2.0
119,1,0.973584,0.822303,0.973395,0.923094,3.0
120,1,0.974642,0.813309,0.971975,0.919976,4.0
116,1,1.000000,0.933512,0.809162,0.914225,5.0
115,1,0.978842,0.773789,0.938917,0.897183,6.0
95,1,0.081709,0.123037,0.167867,0.124204,26.0
109,1,0.148738,0.079297,0.022032,0.083356,37.0
47,1,0.143503,0.050548,0.027496,0.073849,44.0
113,1,0.031589,0.084083,0.069068,0.061580,56.0


Logistic Regression도 동일한 Nested CV 구조로 threshold까지 검증

In [9]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

outer_probs = np.zeros(
    len(X_array)
)

outer_preds = np.zeros(
    len(X_array),
    dtype=int
)

fold_results = []


# ============================================================
# 2. Nested CV
# ============================================================

for outer_fold, (
    outer_train_idx,
    outer_val_idx
) in enumerate(
    outer_cv.split(
        X_array,
        y_array
    ),
    start=1
):

    print(
        f"\n========== Outer Fold {outer_fold} =========="
    )

    X_outer_train = X_array[
        outer_train_idx
    ]

    y_outer_train = y_array[
        outer_train_idx
    ]

    X_outer_val = X_array[
        outer_val_idx
    ]

    y_outer_val = y_array[
        outer_val_idx
    ]


    # ========================================================
    # 3. Inner CV
    # ========================================================

    inner_cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + outer_fold
    )

    inner_oof_prob = np.zeros(
        len(X_outer_train)
    )


    for (
        inner_train_idx,
        inner_val_idx
    ) in inner_cv.split(
        X_outer_train,
        y_outer_train
    ):

        X_inner_train = X_outer_train[
            inner_train_idx
        ]

        y_inner_train = y_outer_train[
            inner_train_idx
        ]

        X_inner_val = X_outer_train[
            inner_val_idx
        ]


        model = LogisticRegression(
            max_iter=1000,
            random_state=42
        )

        model.fit(
            X_inner_train,
            y_inner_train
        )


        inner_oof_prob[
            inner_val_idx
        ] = model.predict_proba(
            X_inner_val
        )[:, 1]


    # ========================================================
    # 4. Inner OOF에서 F1 최대 Threshold 선택
    # ========================================================

    candidate_thresholds = np.unique(
        inner_oof_prob
    )

    best_threshold = None
    best_f1 = -1


    for threshold in candidate_thresholds:

        inner_pred = (
            inner_oof_prob >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_outer_train,
            inner_pred,
            zero_division=0
        )

        if current_f1 > best_f1:

            best_f1 = current_f1
            best_threshold = threshold


    print(
        "Selected Threshold:",
        best_threshold
    )

    print(
        "Inner F1:",
        best_f1
    )


    # ========================================================
    # 5. 전체 Outer Train으로 Logistic 재학습
    # ========================================================

    final_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    final_model.fit(
        X_outer_train,
        y_outer_train
    )


    # ========================================================
    # 6. Outer Validation 확률
    # ========================================================

    val_prob = final_model.predict_proba(
        X_outer_val
    )[:, 1]

    val_pred = (
        val_prob >= best_threshold
    ).astype(int)


    outer_probs[
        outer_val_idx
    ] = val_prob

    outer_preds[
        outer_val_idx
    ] = val_pred


    # ========================================================
    # 7. Fold 성능
    # ========================================================

    tn, fp, fn, tp = confusion_matrix(
        y_outer_val,
        val_pred,
        labels=[0, 1]
    ).ravel()


    fold_results.append({

        "Fold": outer_fold,

        "Threshold":
            best_threshold,

        "Inner_F1":
            best_f1,

        "Outer_Precision":
            precision_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_Recall":
            recall_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "Outer_F1":
            f1_score(
                y_outer_val,
                val_pred,
                zero_division=0
            ),

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


# ============================================================
# 8. Fold별 결과
# ============================================================

nested_logistic_df = pd.DataFrame(
    fold_results
)

print(
    "\n[Nested Logistic Fold Results]"
)

display(
    nested_logistic_df
)


# ============================================================
# 9. 전체 Outer OOF 성능
# ============================================================

precision = precision_score(
    y_array,
    outer_preds,
    zero_division=0
)

recall = recall_score(
    y_array,
    outer_preds,
    zero_division=0
)

f1 = f1_score(
    y_array,
    outer_preds,
    zero_division=0
)

pr_auc = average_precision_score(
    y_array,
    outer_probs
)

tn, fp, fn, tp = confusion_matrix(
    y_array,
    outer_preds,
    labels=[0, 1]
).ravel()


print(
    "\n[Nested Logistic Overall]"
)

print(
    "PR-AUC   :", pr_auc
)

print(
    "Precision:", precision
)

print(
    "Recall   :", recall
)

print(
    "F1       :", f1
)

print(
    "TP       :", tp
)

print(
    "FP       :", fp
)

print(
    "FN       :", fn
)

print(
    "TN       :", tn
)


========== Outer Fold 1 ==========
Selected Threshold: 0.35742610185002976
Inner F1: 0.5

========== Outer Fold 2 ==========
Selected Threshold: 0.894253180270133
Inner F1: 0.5263157894736842

========== Outer Fold 3 ==========
Selected Threshold: 0.9248779051099834
Inner F1: 0.5263157894736842

========== Outer Fold 4 ==========
Selected Threshold: 0.9118788369317687
Inner F1: 0.6

========== Outer Fold 5 ==========
Selected Threshold: 0.8542306620621496
Inner F1: 0.47058823529411764

[Nested Logistic Fold Results]


,Fold,Threshold,Inner_F1,Outer_Precision,Outer_Recall,Outer_F1,TP,FP,FN,TN
0,1,0.357426,0.500000,0.4,0.500000,0.444444,2,3,2,236
1,2,0.894253,0.526316,1.0,0.333333,0.500000,1,0,2,239
2,3,0.924878,0.526316,1.0,0.333333,0.500000,1,0,2,239
3,4,0.911879,0.600000,0.0,0.000000,0.000000,0,0,3,239
4,5,0.854231,0.470588,1.0,0.500000,0.666667,2,0,2,238



[Nested Logistic Overall]
PR-AUC   : 0.46941459686939674
Precision: 0.6666666666666666
Recall   : 0.35294117647058826
F1       : 0.46153846153846156
TP       : 6
FP       : 3
FN       : 11
TN       : 1191
